In [1]:
import pandas as pd
from datetime import datetime
import duckdb

Задача: «Лидеры категорий»
Контекст:
Маркетингу нужно выделить лояльных пользователей, которые тратят в конкретной категории значительно больше, чем «средний» покупатель в этой же категории.

Твое задание:
Выведи список пользователей, чьи суммарные траты в конкретной категории выше средних суммарных трат всех пользователей в этой же категории.

Таблицы:

orders (информация о заказах):

order_id (int) — идентификатор заказа;

user_id (int) — идентификатор пользователя;

order_date (date) — дата заказа.

order_items (детализация заказов):

order_id (int) — идентификатор заказа;

product_id (int) — идентификатор товара;

quantity (int) — количество купленного товара.

products (справочник товаров):

product_id (int) — идентификатор товара;

category_name (varchar) — название категории;

price (numeric) — цена за единицу товара.

In [2]:

orders = pd.DataFrame({
    'order_id': [1, 2, 3, 4, 5, 6, 7, 8],
    'user_id': [101, 102, 101, 103, 102, 104, 103, 104],
    'order_date': [
        datetime(2023, 1, 15),
        datetime(2023, 1, 16),
        datetime(2023, 1, 17),
        datetime(2023, 1, 18),
        datetime(2023, 1, 19),
        datetime(2023, 1, 20),
        datetime(2023, 1, 21),
        datetime(2023, 1, 22)
    ]
})
order_items = pd.DataFrame({
    'order_id': [1, 1, 2, 3, 3, 4, 5, 6, 7, 8],
    'product_id': [1001, 1002, 1003, 1001, 1004, 1005, 1006, 1007, 1008, 1009],
    'quantity': [2, 1, 3, 1, 2, 4, 2, 1, 3, 2]
})
products = pd.DataFrame({
    'product_id': [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009],
    'category_name': ['Электроника', 'Электроника', 'Одежда', 'Электроника',
                    'Одежда', 'Продукты', 'Продукты', 'Книги', 'Книги'],
    'price': [15000, 8000, 5000, 12000, 3000, 200, 150, 400, 600]
})

In [9]:
query = """
WITH user_spending AS (
    SELECT
        p.category_name,
        o.user_id,
        SUM(oi.quantity * p.price) AS total_user_category_spent
    FROM orders o
    JOIN order_items oi USING(order_id)
    JOIN products p USING(product_id)
    GROUP BY 1, 2
)
SELECT 
    user_id,
    category_name,
    total_user_category_spent
FROM (
    SELECT *, 
           AVG(total_user_category_spent) OVER(PARTITION BY category_name) as avg_spent_cat
    FROM user_spending
) t
WHERE total_user_category_spent > avg_spent_cat;
"""
result = duckdb.query(query).to_df()
result

,user_id,category_name,total_user_category_spent
0,102,Одежда,15000.0
1,102,Продукты,400.0


In [22]:
df = orders.merge(order_items).merge(products)
df['summ'] = df['quantity'] * df['price']
df = df.groupby(['category_name', 'user_id'], as_index=False).agg(total_user_category_spent=('summ', 'sum'))
df['avg_spent_cat'] = df.groupby('category_name').total_user_category_spent.transform('mean')
df[df['total_user_category_spent'] > df['avg_spent_cat']][['user_id', 'category_name', 'total_user_category_spent']]

,user_id,category_name,total_user_category_spent
2,102,Одежда,15000
4,102,Продукты,400


In [ ]:
# 1. Заранее оставляем только нужные колонки
items = order_items[['order_id', 'product_id', 'quantity', 'price']]
prods = products[['product_id', 'category_name']]
ords = orders[['order_id', 'user_id']]

# 2. Считаем стоимость сразу в order_items, чтобы не тащить лишнее через мерджи
items['summ'] = items['quantity'] * items['price']

# 3. Последовательно объединяем и группируем
# Здесь мы избавляемся от таблицы orders как можно раньше
res = items.merge(prods, on='product_id').merge(ords, on='order_id')

# 4. Группировка
df = res.groupby(['category_name', 'user_id'], as_index=False)['summ'].sum()
df.rename(columns={'summ': 'total_user_category_spent'}, inplace=True)

# 5. Оконная функция и фильтр
df['avg_spent_cat'] = df.groupby('category_name')['total_user_category_spent'].transform('mean')
result = df[df['total_user_category_spent'] > df['avg_spent_cat']]

Задача: «Retention первого месяца»
Контекст:
Нам нужно понять, насколько качественный трафик пришел к нам в январе 2024 года. Для этого необходимо рассчитать Retention 1-го месяца (какой процент пользователей, зарегистрировавшихся в январе, совершил хотя бы одно действие в феврале).

Твое задание:
Напиши запрос, который выведет одну строку с тремя колонками:

Количество пользователей, зарегистрировавшихся в январе 2024 года.

Количество этих же пользователей, которые совершили хотя бы одно действие в феврале 2024 года.

Процентное соотношение (Retention Rate), округленное до двух знаков после запятой.

Таблицы:

users (регистрации):

user_id (int) — идентификатор пользователя;

signup_date (date) — дата регистрации.

user_actions (активность):

action_id (int) — идентификатор действия;

user_id (int) — идентификатор пользователя;

action_date (date) — дата совершения действия.

Ожидаемый результат:
Колонки: jan_cohort_size, feb_returned_users, retention_rate.

In [14]:
users_data = {
    'user_id': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'signup_date': [
        '2024-01-05', '2024-01-12', '2024-01-25', '2024-01-30', '2024-01-31', # Январские
        '2023-12-25', '2023-12-30',                                           # Декабрьские
        '2024-02-01', '2024-02-10', '2024-02-15'                              # Февральские
    ]
}
users = pd.DataFrame(users_data)
users['signup_date'] = pd.to_datetime(users['signup_date'])
actions_data = {
    'action_id': range(1, 16),
    'user_id': [
        1, 1, 2, 4, 4,  # Активность январских в январе
        1, 2, 2,        # Активность январских в феврале (наш таргет)
        6, 7,           # Активность декабрьских в январе
        8, 9, 10,       # Активность февральских в феврале
        1, 4            # Активность январских в марте
    ],
    'action_date': [
        '2024-01-06', '2024-01-10', '2024-01-15', '2024-01-30', '2024-01-31',
        '2024-02-05', '2024-02-10', '2024-02-20',
        '2024-01-10', '2024-01-15',
        '2024-02-02', '2024-02-11', '2024-02-16',
        '2024-03-01', '2024-03-05'
    ]
}
user_actions = pd.DataFrame(actions_data)
user_actions['action_date'] = pd.to_datetime(user_actions['action_date'])


In [8]:
query = """
SELECT 
    COUNT(distinct user_id) AS jan_cohort_size,
    COUNT(DISTINCT CASE WHEN action_date >= '2024-02-01' AND action_date < '2024-03-01' THEN user_id END) AS feb_returned_users,
    ROUND(
        COUNT(DISTINCT CASE WHEN action_date >= '2024-02-01' AND action_date < '2024-03-01' THEN user_id END) * 100.0 / 
        NULLIF(COUNT(distinct user_id), 0), 
    2) AS retention_rate
FROM users u
LEFT JOIN user_actions a USING(user_id)
WHERE u.signup_date >= '2024-01-01' AND u.signup_date < '2024-02-01';
"""
result = duckdb.query(query).to_df()
result

,jan_cohort_size,feb_returned_users,retention_rate
0,5,2,40.0


In [29]:
jan_cohort = users[
    (users['signup_date'] >= '2024-01-01') & 
    (users['signup_date'] < '2024-02-01')
]
jan_cohort_size = jan_cohort['user_id'].nunique()
merged_data = jan_cohort.merge(user_actions, how='left')
feb_returned_users = merged_data[
    (merged_data['action_date'] >= '2024-02-01') & 
    (merged_data['action_date'] < '2024-03-01')
]['user_id'].nunique()
retention_rate = round((feb_returned_users * 100.0 / jan_cohort_size), 2) if jan_cohort_size > 0 else 0
result = pd.DataFrame({
    'jan_cohort_size': [jan_cohort_size],
    'feb_returned_users': [feb_returned_users],
    'retention_rate': [retention_rate]
})
result

,jan_cohort_size,feb_returned_users,retention_rate
0,5,2,40.0


«Топ-3 в категории»
Раз уж ты упомянул, что тебе нравится визуализировать шаги и мы работаем с оконными функциями, давай закрепим этот навык на классической задаче про ранжирование.

Контекст:
Нам нужно понять, какие товары приносят больше всего денег в каждой категории, чтобы вынести их на главную страницу сайта.

Твое задание:
Для каждой категории товаров выведи топ-3 товара по суммарной выручке за всё время.

Таблицы:

order_items:

product_id (int);

quantity (int);

price_at_purchase (numeric) — цена товара в момент покупки.

products:

product_id (int);

product_name (varchar);

category_id (int).

categories:

category_id (int);

category_name (varchar).

Ожидаемый результат:
Колонки: category_name, product_name, total_revenue, rank.
Результат должен быть отсортирован по названию категории, а внутри категории — по убыванию выручки.

In [2]:
categories_data = {
    'category_id': [1, 2, 3, 4],
    'category_name': ['Электроника', 'Одежда', 'Дом и сад', 'Книги']
}
categories = pd.DataFrame(categories_data)
products_data = {
    'product_id': [101, 102, 103, 104, 105, 201, 202, 203, 301, 302, 401],
    'product_name': [
        'Смартфон X', 'Ноутбук Y', 'Наушники Z', 'Зарядка A', 'Планшет B', # Электроника
        'Футболка', 'Джинсы', 'Худи',                                     # Одежда
        'Лампа', 'Диван',                                                # Дом
        'SQL для профи'                                                  # Книги
    ],
    'category_id': [1, 1, 1, 1, 1, 2, 2, 2, 3, 3, 4]
}
products = pd.DataFrame(products_data)
order_items_data = {
    'product_id': [
        101, 101, 102, 103, 104, 105, # Электроника: Смартфон (101) и Ноутбук (102) будут лидерами
        201, 202, 202, 203,           # Одежда: Джинсы (202) обгонят футболку
        301, 302, 302,                # Дом: Диван (302) лидер по цене
        401, 401                      # Книги: Всего один товар в категории
    ],
    'quantity': [
        2, 1, 1, 5, 10, 1, 
        3, 2, 1, 1, 
        4, 1, 1, 
        10, 5
    ],
    'price_at_purchase': [
        50000, 50000, 120000, 3000, 500, 30000, 
        1500, 4000, 4000, 6000, 
        2000, 45000, 45000, 
        800, 800
    ]
}
order_items = pd.DataFrame(order_items_data)

In [5]:
query = """
select
    category_name,
    product_name,
    total_revenue,
    rank
from (
    select
        c.category_name,
        p.product_name,
        total_revenue,
        dense_rank() over(partition by c.category_name order by total_revenue desc) as rank
    from 
        categories c join products p using(category_id)
        join (
            select
                product_id,
                sum(quantity * price_at_purchase) as total_revenue
            from order_items
            group by product_id
              ) using(product_id)
      )
where rank <= 3
order by category_name, total_revenue desc
"""
result = duckdb.query(query).to_df()
result

,category_name,product_name,total_revenue,rank
0,Дом и сад,Диван,90000.0,1
1,Дом и сад,Лампа,8000.0,2
2,Книги,SQL для профи,12000.0,1
3,Одежда,Джинсы,12000.0,1
4,Одежда,Худи,6000.0,2
5,Одежда,Футболка,4500.0,3
6,Электроника,Смартфон X,150000.0,1
7,Электроника,Ноутбук Y,120000.0,2
8,Электроника,Планшет B,30000.0,3


In [10]:
order_items['revenue'] = order_items['quantity'] * order_items['price_at_purchase']
df = order_items.groupby('product_id', as_index=False).agg(total_revenue=('revenue', 'sum'))
df = df.merge(products).merge(categories)
df['rank'] = df.groupby('category_id')['total_revenue'].rank(method='dense', ascending=False)
df[df['rank'] <= 3].sort_values(by=['category_name', 'total_revenue'], ascending=[True, False])[['category_name', 'product_name', 'total_revenue', 'rank']]


,category_name,product_name,total_revenue,rank
9,Дом и сад,Диван,90000,1.0
8,Дом и сад,Лампа,8000,2.0
10,Книги,SQL для профи,12000,1.0
6,Одежда,Джинсы,12000,1.0
7,Одежда,Худи,6000,2.0
5,Одежда,Футболка,4500,3.0
0,Электроника,Смартфон X,150000,1.0
1,Электроника,Ноутбук Y,120000,2.0
4,Электроника,Планшет B,30000,3.0


«Скорость второго заказа»
Контекст:
Бизнесу важно понимать, как быстро пользователи возвращаются за второй покупкой. Нам нужно вычислить среднее количество дней между первой и второй покупкой для каждого пользователя.

Твое задание:
Для каждого пользователя, у которого было как минимум два заказа, найди разницу в днях между его первым и вторым заказом. В итоге выведи среднее значение этой разницы по всем таким пользователям.

Таблица orders:

order_id (int)

user_id (int)

order_date (date)

Ожидаемый результат:
Одна колонка (или одно значение) — avg_days_to_second_order, округленное до одного знака.

In [13]:
import pandas as pd
orders_data = {
    'order_id': range(1, 15), 
    'user_id': [
        10, 10, 10,  # Юзер 10 (3 заказа)
        20, 20,      # Юзер 20 (2 заказа)
        30,          # Юзер 30 (1 заказ)
        40, 40,      # Юзер 40 (2 заказа)
        50, 50, 50,  # Юзер 50 (3 заказа)
        60,          # Юзер 60 (1 заказ)
        70, 70       # Юзер 70 (2 заказа)
    ],
    'order_date': [
        '2024-01-01', '2024-01-05', '2024-01-10',
        '2024-01-02', '2024-01-15',
        '2024-01-10',
        '2024-01-01', '2024-02-01',
        '2024-02-01', '2024-02-02', '2024-02-15',
        '2024-02-10',
        '2024-03-01', '2024-03-10'
    ]
}
orders = pd.DataFrame(orders_data)
orders['order_date'] = pd.to_datetime(orders['order_date'])
orders = orders.sample(frac=1, random_state=42).reset_index(drop=True)

In [16]:
query = """
with row as(
    select 
        user_id,
        order_date,
        row_number() over(partition by user_id order by order_date) as row_date
    from orders
    where user_id in (select user_id from orders group by user_id)       
)
select 
    avg(r_2.order_date - r_1.order_date) as avg_days_to_second_order
from row r_1 join row r_2 
    on r_1.user_id = r_2.user_id
    and r_1.row_date = 1
    and r_2.row_date = 2
"""
result = duckdb.query(query).to_df()
result

,avg_days_to_second_order
0,11 days 14:24:00


In [18]:
query = """
WITH orders_with_next AS (
    SELECT 
        user_id,
        order_date,
        LEAD(order_date) OVER(PARTITION BY user_id ORDER BY order_date) as next_order_date,
        ROW_NUMBER() OVER(PARTITION BY user_id ORDER BY order_date) as rn
    FROM orders
)
SELECT 
    AVG(next_order_date - order_date) as avg_days_to_second_order
FROM orders_with_next
WHERE rn = 1 AND next_order_date IS NOT NULL;
"""
result = duckdb.query(query).to_df()
result

,avg_days_to_second_order
0,11 days 14:24:00


In [37]:
orders['rank'] = orders.groupby('user_id')['order_date'].rank(method='dense')
df = orders.pivot(
    index='user_id',
    columns='rank',
    values='order_date'
).reset_index()
df = df.iloc[:,[0, 1, 2]].dropna()
avg_days_to_second_order = (df[2.0] - df[1.0]).mean()
result = pd.DataFrame({'avg_days_to_second_order': [avg_days_to_second_order]})
result

,avg_days_to_second_order
0,11 days 14:24:00


In [ ]:
orders_sorted = orders.sort_values(['user_id', 'order_date'])
orders_sorted['prev_date'] = orders_sorted.groupby('user_id')['order_date'].shift(1)
orders_sorted['days_diff'] = (orders_sorted['order_date'] - orders_sorted['prev_date']).dt.days

second_orders_mask = orders_sorted.groupby('user_id').cumcount() == 1
avg_days = orders_sorted[second_orders_mask]['days_diff'].mean()
avg_days

np.float64(11.6)